# Fake News Detection from Image Text and Captions

This notebook builds a reproducible baseline classifier from images in `images/real` and `images/fake`. The folder name supplies the label—there is no separate annotation step.

**Audience:** Python users who can run notebook cells.  
**Goal:** extract OCR text, generate an image caption, compare their meanings with SBERT, then evaluate a Logistic Regression classifier.

Run the cells from top to bottom. The first complete feature-extraction run can take a long time; results are checkpointed in `extracted_features.csv`.

## Outline

1. Install the packages.
2. Import libraries and configure paths.
3. Build a table of image paths and labels.
4. Load OCR, captioning, and embedding models.
5. Extract and save features.
6. Train and evaluate the classifier.

## Step 1 — Install required packages

Run this once in the notebook kernel and let it finish. If the next cell reports an import error, restart the kernel once, then continue.

In [1]:
%pip install easyocr transformers torch sentence-transformers scikit-learn pandas pillow huggingface_hub matplotlib nltk xgboost

Note: you may need to restart the kernel to use updated packages.


## Step 2 — Import libraries and configure paths

`BASE_PATH` must point to the folder containing the `real` and `fake` folders. Feature lengths below are character counts.

In [2]:
from pathlib import Path
import os
import warnings

import easyocr
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sentence_transformers import SentenceTransformer, util
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from transformers import BlipForConditionalGeneration, BlipProcessor

warnings.filterwarnings('ignore', category=FutureWarning)

BASE_PATH = Path(r'E:\Project\Fake News Detection\images')
LABELS = ['real', 'fake']
FEATURES_CSV = Path('extracted_features.csv')
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

assert BASE_PATH.is_dir(), f'Image folder not found: {BASE_PATH}'
print(f'Using device: {DEVICE}')
print(f'Base path: {BASE_PATH}')


Using device: cpu
Base path: E:\Project\Fake News Detection\images


## Step 3 — Load image paths and labels

Each image inherits its label from its immediate folder. This accepts `.jpg`, `.jpeg`, `.png`, `.webp`, and `.bmp` files.

In [4]:
records = []

for label in LABELS:
    label_dir = BASE_PATH / label
    assert label_dir.is_dir(), f'Missing label folder: {label_dir}'

    for image_path in sorted(label_dir.iterdir()):
        if image_path.is_file() and image_path.suffix.lower() in IMAGE_SUFFIXES:
            records.append({'image_path': str(image_path.resolve()), 'label': label})

df = pd.DataFrame(records, columns=['image_path', 'label'])
assert not df.empty, f'No supported image files found below {BASE_PATH}'
assert set(df['label']) == set(LABELS), 'At least one label folder has no images.'

print(f'Total images: {len(df):,}')
df['label'].value_counts()


Total images: 2,000


label
real    1000
fake    1000
Name: count, dtype: int64

## Step 4 — Download and load the models

The first run downloads EasyOCR weights, the BLIP image-captioning model, and SBERT (roughly 1–1.5 GB in total). This may take 10–20 minutes. Re-run this cell only after restarting the kernel.

In [5]:
import sys
sys.path.insert(0, str(Path.cwd()))
from news_guard.features import extract_ocr_text, preprocess_for_ocr, OCR_LANGUAGES

BLIP_MODEL_ID = 'Salesforce/blip-image-captioning-base'
SBERT_MODEL_ID = 'all-MiniLM-L6-v2'

# OCR_LANGUAGES is now ['bn', 'en']: this dataset is Bangladeshi news imagery
# (BANGLADESH BANK, TAKA, BUET, ...) and an English-only reader forced Bengali
# glyphs into nonsense Latin/digit lookalikes. extract_ocr_text also applies
# grayscale/autocontrast/upscale preprocessing and drops low-confidence text.
reader = easyocr.Reader(OCR_LANGUAGES, model_storage_directory=str(Path('models') / 'easyocr'))
blip_processor = BlipProcessor.from_pretrained(BLIP_MODEL_ID)
blip_model = BlipForConditionalGeneration.from_pretrained(BLIP_MODEL_ID).to(DEVICE)
blip_model.eval()
embedding_model = SentenceTransformer(SBERT_MODEL_ID, device=DEVICE)

print('All models loaded.')


Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.
e:\Project\Fake News Detection\.venv\Lib\site-packages\torch\ao\nn\quantized\dynamic\modules\rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\quantized\Quantizer.cpp:116.)
  w_ih = torch.quantize_per_tensor(


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

All models loaded.


## Step 5 — Define feature extraction

OCR reads any visible English text. BLIP makes an independent visual caption. SBERT cosine similarity measures how closely the OCR text and caption agree; it is only one signal and does not itself prove whether a story is true.

In [6]:
# extract_ocr_text is imported from news_guard.features (bn+en, preprocessed,
# confidence-filtered) so the notebook and the live app never drift apart.


def generate_caption(image: Image.Image) -> str:
    inputs = blip_processor(images=image, return_tensors='pt').to(DEVICE)
    with torch.inference_mode():
        generated_ids = blip_model.generate(**inputs, max_new_tokens=40)
    return blip_processor.decode(generated_ids[0], skip_special_tokens=True).strip()


def sbert_similarity(ocr_text: str, caption: str) -> float:
    if not ocr_text or not caption:
        return float('nan')

    embeddings = embedding_model.encode(
        [ocr_text, caption],
        convert_to_tensor=True,
        normalize_embeddings=True,
    )
    return float(util.cos_sim(embeddings[0], embeddings[1]).item())


## Step 6 — Run extraction and save a checkpoint

This processes every image not already present in `extracted_features.csv`, saves progress every 25 images, and preserves OCR/caption text for inspection. If the runtime stops, run this cell again to resume.

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from PIL import Image
from sentence_transformers import util

# Uses: df, reader, blip_processor, blip_model,
# embedding_model, DEVICE from earlier notebook cells.

FEATURES_CSV = Path("extracted_features.csv")
FEATURE_COLUMNS = [
    "image_path", "label", "ocr_text", "caption",
    "ocr_length", "caption_length", "similarity_score", "error",
]

# extract_ocr_text is imported from news_guard.features (bn+en, preprocessed,
# confidence-filtered) so this checkpoint loop matches the live app exactly.
def generate_caption(image):
    inputs = blip_processor(images=image, return_tensors="pt").to(DEVICE)

    with torch.inference_mode():
        generated_ids = blip_model.generate(
            **inputs,
            max_new_tokens=40
        )

    return blip_processor.decode(
        generated_ids[0],
        skip_special_tokens=True
    ).strip()

def sbert_similarity(ocr_text, caption):
    # No visible OCR text means cosine similarity is not meaningful.
    if not ocr_text or not caption:
        return float("nan")

    embeddings = embedding_model.encode(
        [ocr_text, caption],
        convert_to_tensor=True,
        normalize_embeddings=True,
    )

    return float(util.cos_sim(embeddings[0], embeddings[1]).item())

def save_checkpoint(dataframe):
    """Safely replaces the CSV and waits if Excel has locked it."""
    temp_csv = FEATURES_CSV.with_name("extracted_features.tmp.csv")
    dataframe.to_csv(temp_csv, index=False)

    while True:
        try:
            os.replace(temp_csv, FEATURES_CSV)
            return
        except PermissionError:
            print("Close extracted_features.csv in Excel. Retrying in 5 seconds...")
            time.sleep(5)

# Load previous completed rows so the notebook resumes instead of starting again.
if FEATURES_CSV.exists():
    features_df = pd.read_csv(FEATURES_CSV)

    if set(FEATURE_COLUMNS).issubset(features_df.columns):
        features_df = features_df[FEATURE_COLUMNS].copy()
    else:
        features_df = pd.DataFrame(columns=FEATURE_COLUMNS)
else:
    features_df = pd.DataFrame(columns=FEATURE_COLUMNS)

completed_paths = set(features_df["image_path"].dropna().astype(str))
pending_df = df.loc[
    ~df["image_path"].isin(completed_paths)
].copy()

print(f"Already processed: {len(features_df):,}")
print(f"Remaining images: {len(pending_df):,}")

new_rows = []

for number, record in enumerate(pending_df.to_dict("records"), start=1):
    try:
        with Image.open(record["image_path"]) as opened_image:
            image = opened_image.convert("RGB")

        ocr_text = extract_ocr_text(image)
        caption = generate_caption(image)
        similarity_score = sbert_similarity(ocr_text, caption)
        error = ""

    except Exception as exc:
        ocr_text = ""
        caption = ""
        similarity_score = float("nan")
        error = f"{type(exc).__name__}: {exc}"

    new_rows.append({
        "image_path": record["image_path"],
        "label": record["label"],
        "ocr_text": ocr_text,
        "caption": caption,
        "ocr_length": len(ocr_text),
        "caption_length": len(caption),
        "similarity_score": similarity_score,
        "error": error,
    })

    # Save every 10 images, so interruption does not lose progress.
    if number % 10 == 0 or number == len(pending_df):
        features_df = pd.concat(
            [features_df, pd.DataFrame(new_rows)],
            ignore_index=True
        )

        features_df = features_df.drop_duplicates(
            subset="image_path",
            keep="last"
        )

        save_checkpoint(features_df)
        new_rows = []

        print(f"Processed {number:,}/{len(pending_df):,} remaining images")

print(f"\nFinished feature processing.")
print(f"Total rows saved: {len(features_df):,}")
print(f"Failed images: {(features_df['error'].fillna('') != '').sum():,}")

## Step 7 — Train a classifier and report metrics

The model uses the three numeric features requested: `similarity_score`, `ocr_length`, and `caption_length`. The scaler is fitted only on the training split through a pipeline, preventing test-set leakage. Precision, recall, and F1 treat `fake` as the positive class.

In [ ]:
features_df = pd.read_csv(FEATURES_CSV)
MODEL_FEATURES = ['similarity_score', 'ocr_length', 'caption_length']

training_df = features_df.dropna(subset=MODEL_FEATURES + ['label']).copy()
training_df = training_df.loc[training_df['error'].fillna('').eq('')].copy()
assert training_df['label'].nunique() == 2, 'Both real and fake examples are required.'
assert training_df['label'].value_counts().min() >= 2, 'Each class needs at least two valid rows.'

X = training_df[MODEL_FEATURES]
y = training_df['label']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)

classifier = Pipeline([
    ('scale', StandardScaler()),
    ('model', LogisticRegression(max_iter=1_000, random_state=SEED)),
])
classifier.fit(X_train, y_train)
predictions = classifier.predict(X_test)

metrics = {
    'accuracy': accuracy_score(y_test, predictions),
    'precision (fake)': precision_score(y_test, predictions, pos_label='fake', zero_division=0),
    'recall (fake)': recall_score(y_test, predictions, pos_label='fake', zero_division=0),
    'F1 (fake)': f1_score(y_test, predictions, pos_label='fake', zero_division=0),
}

for name, value in metrics.items():
    print(f'{name:18s}: {value:.4f}')

matrix = confusion_matrix(y_test, predictions, labels=LABELS)
ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=LABELS).plot(cmap='Blues')
print('Rows used for training:', len(training_df))


## Step 8 — CISF-style cross-image semantic fusion

This adds a practical Cross-Image Semantic Fusion (CISF) module. For each image, SBERT embeds its OCR text and BLIP caption together, retrieves the most semantically similar *other* images, and fuses their embeddings with the image's own embedding. It then classifies using the baseline numeric features plus CISF neighbour-consistency features and PCA-reduced fused features.

This is a CISF-style implementation for this folder-labelled dataset, not a claim of an exact reproduction of Wang et al. (MSN 2024). Crucially, neighbours for test images come only from training images, and labels are never used to choose neighbours. Run it only after Step 6 has completed all images.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

CISF_NEIGHBORS = 5
CISF_ALPHA = 0.65  # weight kept from the target image; 0.35 comes from neighbours
CISF_COMPONENTS = 32
CISF_OUTPUT_CSV = Path('cisf_features.csv')

# Use rows that have a label, a caption/OCR representation, and no extraction error.
features_df = pd.read_csv(FEATURES_CSV)
cisf_df = features_df.loc[
    features_df['error'].fillna('').eq('') & features_df['label'].isin(LABELS)
].copy().reset_index(drop=True)

assert cisf_df['label'].nunique() == 2, 'Finish feature extraction for both real and fake images first.'
assert cisf_df['label'].value_counts().min() >= 3, 'Each class needs at least three valid images.'

# Captions give an embedding even when an image contains no readable OCR text.
cisf_df['semantic_text'] = (
    'caption: ' + cisf_df['caption'].fillna('').astype(str).str.strip()
    + ' [SEP] ocr: ' + cisf_df['ocr_text'].fillna('').astype(str).str.strip()
)

semantic_embeddings = embedding_model.encode(
    cisf_df['semantic_text'].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

all_indices = np.arange(len(cisf_df))
train_indices, test_indices = train_test_split(
    all_indices,
    test_size=0.20,
    stratify=cisf_df['label'].to_numpy(),
    random_state=SEED,
)

train_embeddings = semantic_embeddings[train_indices]
test_embeddings = semantic_embeddings[test_indices]


def cisf_fuse(query_embeddings, candidate_embeddings, exclude_self=False):
    """Fuse each query with its nearest semantic candidate images."""
    requested = CISF_NEIGHBORS + int(exclude_self)
    n_neighbors = min(requested, len(candidate_embeddings))
    if n_neighbors < 1:
        raise ValueError('CISF needs at least one candidate image.')

    search = NearestNeighbors(n_neighbors=n_neighbors, metric='cosine', algorithm='brute')
    search.fit(candidate_embeddings)
    distances, positions = search.kneighbors(query_embeddings)

    fused_vectors = []
    mean_similarities = []
    max_similarities = []

    for row_number, (query, row_distances, row_positions) in enumerate(
        zip(query_embeddings, distances, positions)
    ):
        similarities = 1.0 - row_distances

        # During training, remove the image itself from its neighbour list.
        if exclude_self:
            keep = row_positions != row_number
            row_positions = row_positions[keep]
            similarities = similarities[keep]

        row_positions = row_positions[:CISF_NEIGHBORS]
        similarities = similarities[:CISF_NEIGHBORS]
        if len(row_positions) == 0:
            raise ValueError('No non-self neighbour was available for CISF fusion.')

        # Similar images contribute more; equal weights are used if all scores are non-positive.
        weights = np.clip(similarities, 0.0, None)
        if weights.sum() == 0:
            weights = np.ones_like(weights)
        weights = weights / weights.sum()

        neighbour_vector = (candidate_embeddings[row_positions] * weights[:, None]).sum(axis=0)
        fused_vector = CISF_ALPHA * query + (1.0 - CISF_ALPHA) * neighbour_vector
        fused_vector = fused_vector / np.linalg.norm(fused_vector)

        fused_vectors.append(fused_vector)
        mean_similarities.append(float(similarities.mean()))
        max_similarities.append(float(similarities.max()))

    return np.vstack(fused_vectors), np.array(mean_similarities), np.array(max_similarities)


# Train images fuse only with other train images; test images fuse only with train images.
train_fused, train_neighbour_mean, train_neighbour_max = cisf_fuse(
    train_embeddings, train_embeddings, exclude_self=True
)
test_fused, test_neighbour_mean, test_neighbour_max = cisf_fuse(
    test_embeddings, train_embeddings, exclude_self=False
)

n_components = min(CISF_COMPONENTS, train_fused.shape[1], train_fused.shape[0] - 1)
pca = PCA(n_components=n_components, random_state=SEED)
train_fused_pca = pca.fit_transform(train_fused)
test_fused_pca = pca.transform(test_fused)


def build_cisf_matrix(indices, neighbour_mean, neighbour_max, fused_pca):
    numeric = cisf_df.iloc[indices][
        ['similarity_score', 'ocr_length', 'caption_length']
    ].copy()
    numeric['similarity_score'] = pd.to_numeric(
        numeric['similarity_score'], errors='coerce'
    ).fillna(0.0)
    numeric['ocr_length'] = pd.to_numeric(numeric['ocr_length'], errors='coerce').fillna(0.0)
    numeric['caption_length'] = pd.to_numeric(numeric['caption_length'], errors='coerce').fillna(0.0)
    numeric['has_ocr_text'] = (
        cisf_df.iloc[indices]['ocr_text'].fillna('').astype(str).str.strip().ne('')
    ).astype(float).to_numpy()
    numeric['cisf_neighbour_mean_similarity'] = neighbour_mean
    numeric['cisf_neighbour_max_similarity'] = neighbour_max

    return np.column_stack([numeric.to_numpy(dtype=float), fused_pca])


X_train_cisf = build_cisf_matrix(
    train_indices, train_neighbour_mean, train_neighbour_max, train_fused_pca
)
X_test_cisf = build_cisf_matrix(
    test_indices, test_neighbour_mean, test_neighbour_max, test_fused_pca
)
y_train_cisf = cisf_df.iloc[train_indices]['label']
y_test_cisf = cisf_df.iloc[test_indices]['label']

cisf_classifier = Pipeline([
    ('scale', StandardScaler()),
    ('model', LogisticRegression(max_iter=2_000, random_state=SEED)),
])
cisf_classifier.fit(X_train_cisf, y_train_cisf)
cisf_predictions = cisf_classifier.predict(X_test_cisf)

cisf_metrics = {
    'accuracy': accuracy_score(y_test_cisf, cisf_predictions),
    'precision (fake)': precision_score(
        y_test_cisf, cisf_predictions, pos_label='fake', zero_division=0
    ),
    'recall (fake)': recall_score(
        y_test_cisf, cisf_predictions, pos_label='fake', zero_division=0
    ),
    'F1 (fake)': f1_score(
        y_test_cisf, cisf_predictions, pos_label='fake', zero_division=0
    ),
}

print(f'CISF feature dimensions: {X_train_cisf.shape[1]}')
for name, value in cisf_metrics.items():
    print(f'{name:18s}: {value:.4f}')

cisf_matrix = confusion_matrix(y_test_cisf, cisf_predictions, labels=LABELS)
ConfusionMatrixDisplay(confusion_matrix=cisf_matrix, display_labels=LABELS).plot(cmap='Purples')

# Save compact, inspectable CISF neighbour features for every image used.
cisf_export = cisf_df[['image_path', 'label']].copy()
cisf_export['split'] = 'train'
cisf_export.loc[test_indices, 'split'] = 'test'
cisf_export['cisf_neighbour_mean_similarity'] = np.nan
cisf_export['cisf_neighbour_max_similarity'] = np.nan
cisf_export.loc[train_indices, 'cisf_neighbour_mean_similarity'] = train_neighbour_mean
cisf_export.loc[test_indices, 'cisf_neighbour_mean_similarity'] = test_neighbour_mean
cisf_export.loc[train_indices, 'cisf_neighbour_max_similarity'] = train_neighbour_max
cisf_export.loc[test_indices, 'cisf_neighbour_max_similarity'] = test_neighbour_max
CISF_COMPONENT_COLUMNS = [
    f'cisf_fused_pca_{component_index:02d}'
    for component_index in range(n_components)
]
for component_index, column in enumerate(CISF_COMPONENT_COLUMNS):
    cisf_export[column] = np.nan
    cisf_export.loc[train_indices, column] = train_fused_pca[:, component_index]
    cisf_export.loc[test_indices, column] = test_fused_pca[:, component_index]
cisf_export.to_csv(CISF_OUTPUT_CSV, index=False)
print(f'CISF features saved to: {CISF_OUTPUT_CSV.resolve()}')


## Step 9 — Text preprocessing, Count Vectorizer, and TF-IDF

This step combines the BLIP caption and OCR text, tokenizes it, removes ordinary stop words while keeping negations, stems words to a common root, and creates Count Vectorizer and TF-IDF representations. The train/test split happens **before** fitting either vectorizer, preventing test-set vocabulary leakage.

In [ ]:
import re

from nltk.stem import SnowballStemmer
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import (
    CountVectorizer,
    ENGLISH_STOP_WORDS,
    TfidfVectorizer,
)

TEXT_TEST_SIZE = 0.20
TEXT_MAX_FEATURES = 5_000
TEXT_MIN_DOCUMENT_FREQUENCY = 3

stemmer = SnowballStemmer('english')
# Preserve negation terms because they can change a claim's meaning.
STOP_WORDS = ENGLISH_STOP_WORDS.difference({'no', 'not', 'nor', 'never'})


def tokenize_and_stem(text: str) -> list[str]:
    tokens = re.findall(r"[a-zA-Z][a-zA-Z']+", str(text).lower())
    return [
        stemmer.stem(token)
        for token in tokens
        if token not in STOP_WORDS and len(token) > 1
    ]


features_df = pd.read_csv(FEATURES_CSV)
text_df = features_df.loc[
    features_df['error'].fillna('').eq('') & features_df['label'].isin(LABELS)
].copy().reset_index(drop=True)

assert text_df['label'].nunique() == 2, 'Both real and fake images are required.'

text_df['combined_text'] = (
    'caption: ' + text_df['caption'].fillna('').astype(str).str.strip()
    + ' ocr: ' + text_df['ocr_text'].fillna('').astype(str).str.strip()
)
text_df['processed_text'] = text_df['combined_text'].map(
    lambda value: ' '.join(tokenize_and_stem(value))
)
text_df['token_count'] = text_df['processed_text'].str.split().str.len().fillna(0).astype(int)
text_df['unique_token_ratio'] = text_df['processed_text'].map(
    lambda value: len(set(value.split())) / max(len(value.split()), 1)
)

all_indices = np.arange(len(text_df))
text_train_indices, text_test_indices = train_test_split(
    all_indices,
    test_size=TEXT_TEST_SIZE,
    stratify=text_df['label'].to_numpy(),
    random_state=SEED,
)

train_text = text_df.iloc[text_train_indices]['processed_text']
test_text = text_df.iloc[text_test_indices]['processed_text']
y_train_text = text_df.iloc[text_train_indices]['label']
y_test_text = text_df.iloc[text_test_indices]['label']

count_vectorizer = CountVectorizer(
    tokenizer=str.split,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 2),
    min_df=TEXT_MIN_DOCUMENT_FREQUENCY,
    max_features=TEXT_MAX_FEATURES,
    dtype=np.float32,
)
X_train_count = count_vectorizer.fit_transform(train_text)
X_test_count = count_vectorizer.transform(test_text)

tfidf_vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 2),
    min_df=TEXT_MIN_DOCUMENT_FREQUENCY,
    max_features=TEXT_MAX_FEATURES,
    sublinear_tf=True,
    dtype=np.float32,
)
X_train_tfidf = tfidf_vectorizer.fit_transform(train_text)
X_test_tfidf = tfidf_vectorizer.transform(test_text)

NUMERIC_FEATURE_NAMES = [
    'similarity_score',
    'ocr_length',
    'caption_length',
    'has_ocr_text',
    'token_count',
    'unique_token_ratio',
]


def build_numeric_features(indices: np.ndarray) -> np.ndarray:
    rows = text_df.iloc[indices]
    return np.column_stack([
        pd.to_numeric(rows['similarity_score'], errors='coerce').fillna(0.0),
        pd.to_numeric(rows['ocr_length'], errors='coerce').fillna(0.0),
        pd.to_numeric(rows['caption_length'], errors='coerce').fillna(0.0),
        rows['ocr_text'].fillna('').astype(str).str.strip().ne('').astype(float),
        rows['token_count'].to_numpy(dtype=float),
        rows['unique_token_ratio'].to_numpy(dtype=float),
    ])


X_train_numeric = build_numeric_features(text_train_indices)
X_test_numeric = build_numeric_features(text_test_indices)

print(f'Rows available: {len(text_df):,}')
print(f'Train / test rows: {len(text_train_indices):,} / {len(text_test_indices):,}')
print(f'Count Vectorizer shape: {X_train_count.shape}')
print(f'TF-IDF shape: {X_train_tfidf.shape}')
text_df[['label', 'combined_text', 'processed_text', 'token_count']].head(3)


## Step 10 — Train SVM and XGBoost; export feature importance

Two Linear SVM models compare Count Vectorizer and TF-IDF. XGBoost uses TF-IDF features plus engineered numeric features, then exports its gain-based feature importance to a CSV and chart. All models use the same held-out test split from Step 9.

In [ ]:
import matplotlib.pyplot as plt

from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

MODEL_METRICS_CSV = Path('text_model_metrics.csv')
XGBOOST_IMPORTANCE_CSV = Path('xgboost_feature_importance.csv')
XGBOOST_IMPORTANCE_PNG = Path('xgboost_feature_importance.png')


def calculate_metrics(model_name: str, actual, predicted) -> dict:
    return {
        'model': model_name,
        'accuracy': accuracy_score(actual, predicted),
        'precision_fake': precision_score(
            actual, predicted, pos_label='fake', zero_division=0
        ),
        'recall_fake': recall_score(
            actual, predicted, pos_label='fake', zero_division=0
        ),
        'f1_fake': f1_score(actual, predicted, pos_label='fake', zero_division=0),
    }


count_svm = LinearSVC(C=1.0, random_state=SEED)
count_svm.fit(X_train_count, y_train_text)
count_svm_predictions = count_svm.predict(X_test_count)

tfidf_svm = LinearSVC(C=1.0, random_state=SEED)
tfidf_svm.fit(X_train_tfidf, y_train_text)
tfidf_svm_predictions = tfidf_svm.predict(X_test_tfidf)

# Sparse TF-IDF keeps important word features inspectable for XGBoost.
X_train_xgb = hstack([X_train_tfidf, csr_matrix(X_train_numeric)], format='csr')
X_test_xgb = hstack([X_test_tfidf, csr_matrix(X_test_numeric)], format='csr')
xgb_feature_names = np.concatenate([
    np.array([f'tfidf:{name}' for name in tfidf_vectorizer.get_feature_names_out()]),
    np.array([f'numeric:{name}' for name in NUMERIC_FEATURE_NAMES]),
])

xgb_classifier = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.80,
    colsample_bytree=0.70,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=4,
    tree_method='hist',
    importance_type='gain',
)
xgb_classifier.fit(X_train_xgb, y_train_text.eq('fake').astype(int))
xgb_predictions = np.where(xgb_classifier.predict(X_test_xgb) == 1, 'fake', 'real')

model_metrics = pd.DataFrame([
    calculate_metrics('Linear SVM — Count Vectorizer', y_test_text, count_svm_predictions),
    calculate_metrics('Linear SVM — TF-IDF', y_test_text, tfidf_svm_predictions),
    calculate_metrics('XGBoost — TF-IDF + numeric', y_test_text, xgb_predictions),
]).sort_values('f1_fake', ascending=False).reset_index(drop=True)
model_metrics.to_csv(MODEL_METRICS_CSV, index=False)

xgb_importance = pd.DataFrame({
    'feature': xgb_feature_names,
    'gain_importance': xgb_classifier.feature_importances_,
}).sort_values('gain_importance', ascending=False).reset_index(drop=True)
xgb_importance.to_csv(XGBOOST_IMPORTANCE_CSV, index=False)

top_importance = xgb_importance.head(25).sort_values('gain_importance')
fig, axis = plt.subplots(figsize=(10, 8))
axis.barh(top_importance['feature'], top_importance['gain_importance'], color='#5B5BD6')
axis.set_title('XGBoost: top 25 feature importances (gain)')
axis.set_xlabel('Relative gain importance')
fig.tight_layout()
fig.savefig(XGBOOST_IMPORTANCE_PNG, dpi=160, bbox_inches='tight')
plt.show()

print(model_metrics.to_string(index=False, float_format=lambda value: f'{value:.4f}'))
print(f'\nModel metrics saved to: {MODEL_METRICS_CSV.resolve()}')
print(f'Feature importance CSV saved to: {XGBOOST_IMPORTANCE_CSV.resolve()}')
print(f'Feature importance chart saved to: {XGBOOST_IMPORTANCE_PNG.resolve()}')


## Step 11 — 5-fold cross-validation for SVM and XGBoost

A single train/test split can be lucky or unlucky. This step repeats a stratified train/test evaluation five times. In every fold, TF-IDF is fitted only on that fold's training text, then SVM and XGBoost are evaluated on its held-out images. The final table reports mean and standard deviation across folds.

In [ ]:
from sklearn.model_selection import StratifiedKFold

CROSS_VALIDATION_FOLDS = 5
CROSS_VALIDATION_FOLDS_CSV = Path('cross_validation_fold_metrics.csv')
CROSS_VALIDATION_SUMMARY_CSV = Path('cross_validation_summary.csv')

cv = StratifiedKFold(
    n_splits=CROSS_VALIDATION_FOLDS,
    shuffle=True,
    random_state=SEED,
)

all_processed_text = text_df['processed_text'].to_numpy()
all_labels = text_df['label'].to_numpy()
cross_validation_records = []

for fold_number, (fold_train_indices, fold_test_indices) in enumerate(
    cv.split(all_processed_text, all_labels),
    start=1,
):
    fold_vectorizer = TfidfVectorizer(
        tokenizer=str.split,
        token_pattern=None,
        lowercase=False,
        ngram_range=(1, 2),
        min_df=TEXT_MIN_DOCUMENT_FREQUENCY,
        max_features=TEXT_MAX_FEATURES,
        sublinear_tf=True,
        dtype=np.float32,
    )

    fold_X_train_tfidf = fold_vectorizer.fit_transform(
        all_processed_text[fold_train_indices]
    )
    fold_X_test_tfidf = fold_vectorizer.transform(
        all_processed_text[fold_test_indices]
    )
    fold_y_train = all_labels[fold_train_indices]
    fold_y_test = all_labels[fold_test_indices]

    fold_svm = LinearSVC(C=1.0, random_state=SEED)
    fold_svm.fit(fold_X_train_tfidf, fold_y_train)
    fold_svm_predictions = fold_svm.predict(fold_X_test_tfidf)
    cross_validation_records.append({
        'fold': fold_number,
        **calculate_metrics('Linear SVM - TF-IDF', fold_y_test, fold_svm_predictions),
    })

    fold_X_train_numeric = build_numeric_features(fold_train_indices)
    fold_X_test_numeric = build_numeric_features(fold_test_indices)
    fold_X_train_xgb = hstack(
        [fold_X_train_tfidf, csr_matrix(fold_X_train_numeric)],
        format='csr',
    )
    fold_X_test_xgb = hstack(
        [fold_X_test_tfidf, csr_matrix(fold_X_test_numeric)],
        format='csr',
    )

    fold_xgb = XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.80,
        colsample_bytree=0.70,
        reg_lambda=1.0,
        random_state=SEED,
        n_jobs=4,
        tree_method='hist',
        importance_type='gain',
    )
    fold_xgb.fit(fold_X_train_xgb, pd.Series(fold_y_train).eq('fake').astype(int))
    fold_xgb_predictions = np.where(
        fold_xgb.predict(fold_X_test_xgb) == 1,
        'fake',
        'real',
    )
    cross_validation_records.append({
        'fold': fold_number,
        **calculate_metrics('XGBoost - TF-IDF + numeric', fold_y_test, fold_xgb_predictions),
    })

    print(f'Completed fold {fold_number}/{CROSS_VALIDATION_FOLDS}')

cross_validation_folds = pd.DataFrame(cross_validation_records)
cross_validation_folds.to_csv(CROSS_VALIDATION_FOLDS_CSV, index=False)

cross_validation_summary = (
    cross_validation_folds
    .groupby('model')[['accuracy', 'precision_fake', 'recall_fake', 'f1_fake']]
    .agg(['mean', 'std'])
)
cross_validation_summary.columns = [
    f'{metric}_{statistic}'
    for metric, statistic in cross_validation_summary.columns
]
cross_validation_summary = cross_validation_summary.reset_index().sort_values(
    'f1_fake_mean', ascending=False
)
cross_validation_summary.to_csv(CROSS_VALIDATION_SUMMARY_CSV, index=False)

print('\n5-fold cross-validation summary:')
print(cross_validation_summary.to_string(index=False, float_format=lambda value: f'{value:.4f}'))
print(f'\nPer-fold metrics saved to: {CROSS_VALIDATION_FOLDS_CSV.resolve()}')
print(f'Summary saved to: {CROSS_VALIDATION_SUMMARY_CSV.resolve()}')


## Step 12 — Extract CLIP visual features

CLIP produces a 512-value visual embedding for each image. Unlike OCR, it still represents images without visible text. The output is checkpointed in `clip_visual_features.csv` and resumes if interrupted. This one-time CPU step can take several minutes.

In [ ]:
from transformers import CLIPModel, CLIPProcessor

CLIP_MODEL_ID = 'openai/clip-vit-base-patch32'
CLIP_FEATURES_CSV = Path('clip_visual_features.csv')
CLIP_BATCH_SIZE = 16
CLIP_CHECKPOINT_EVERY = 128

clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_ID).to(DEVICE)
clip_model.eval()
CLIP_DIMENSION = clip_model.config.projection_dim
CLIP_COLUMNS = [f'clip_{index:03d}' for index in range(CLIP_DIMENSION)]
CLIP_FILE_COLUMNS = ['image_path', 'label', 'clip_error'] + CLIP_COLUMNS

source_features = pd.read_csv(FEATURES_CSV)
clip_source = source_features.loc[
    source_features['error'].fillna('').eq('') & source_features['label'].isin(LABELS),
    ['image_path', 'label'],
].drop_duplicates('image_path').reset_index(drop=True)

if CLIP_FEATURES_CSV.exists():
    clip_features_df = pd.read_csv(CLIP_FEATURES_CSV)
    if set(CLIP_FILE_COLUMNS).issubset(clip_features_df.columns):
        clip_features_df = clip_features_df[CLIP_FILE_COLUMNS].copy()
    else:
        print('Existing CLIP CSV has a different schema; starting again.')
        clip_features_df = pd.DataFrame(columns=CLIP_FILE_COLUMNS)
else:
    clip_features_df = pd.DataFrame(columns=CLIP_FILE_COLUMNS)


def save_clip_checkpoint(dataframe: pd.DataFrame) -> None:
    temporary_path = CLIP_FEATURES_CSV.with_name('clip_visual_features.tmp.csv')
    dataframe.to_csv(temporary_path, index=False)
    while True:
        try:
            os.replace(temporary_path, CLIP_FEATURES_CSV)
            return
        except PermissionError:
            print('Close clip_visual_features.csv in Excel. Retrying in 5 seconds...')
            time.sleep(5)


completed_clip_paths = set(clip_features_df['image_path'].dropna().astype(str))
clip_pending = clip_source.loc[
    ~clip_source['image_path'].isin(completed_clip_paths)
].to_dict('records')

print(f'CLIP embeddings already saved: {len(clip_features_df):,}')
print(f'CLIP embeddings remaining: {len(clip_pending):,}')

new_clip_rows = []
for start_index in range(0, len(clip_pending), CLIP_BATCH_SIZE):
    batch_records = clip_pending[start_index:start_index + CLIP_BATCH_SIZE]
    valid_records, batch_images = [], []

    for record in batch_records:
        try:
            with Image.open(record['image_path']) as opened_image:
                batch_images.append(opened_image.convert('RGB'))
            valid_records.append(record)
        except Exception as exc:
            new_clip_rows.append({
                'image_path': record['image_path'],
                'label': record['label'],
                'clip_error': f'{type(exc).__name__}: {exc}',
                **{column: np.nan for column in CLIP_COLUMNS},
            })

    if valid_records:
        clip_inputs = clip_processor(images=batch_images, return_tensors='pt').to(DEVICE)
        with torch.inference_mode():
            image_output = clip_model.get_image_features(**clip_inputs)
            batch_embeddings = image_output.pooler_output
            batch_embeddings = torch.nn.functional.normalize(batch_embeddings, dim=1)
        batch_embeddings = batch_embeddings.cpu().numpy()

        for record, embedding in zip(valid_records, batch_embeddings):
            new_clip_rows.append({
                'image_path': record['image_path'],
                'label': record['label'],
                'clip_error': '',
                **dict(zip(CLIP_COLUMNS, embedding.astype(float))),
            })

    processed_count = min(start_index + CLIP_BATCH_SIZE, len(clip_pending))
    if processed_count % CLIP_CHECKPOINT_EVERY == 0 or processed_count == len(clip_pending):
        clip_features_df = pd.concat(
            [clip_features_df, pd.DataFrame(new_clip_rows)],
            ignore_index=True,
        ).drop_duplicates('image_path', keep='last')
        save_clip_checkpoint(clip_features_df)
        new_clip_rows = []
        print(f'CLIP checkpoint: {processed_count:,}/{len(clip_pending):,}')

print(f'CLIP features saved to: {CLIP_FEATURES_CSV.resolve()}')
print(f'Rows: {len(clip_features_df):,}; failed rows: {(clip_features_df["clip_error"].fillna("") != "").sum():,}')


## Step 13 — Tune multimodal XGBoost with cross-validation

This combines TF-IDF text, numeric image features, CISF neighbour features, and CLIP visual features. A 20% hold-out test set remains unseen. Hyperparameters are selected with 3-fold stratified cross-validation on the other 80%, using fake-news F1 as the optimisation metric. TF-IDF and visual PCA are refitted inside every fold to prevent leakage.

In [ ]:
from scipy.stats import loguniform, randint, uniform
from sklearn.compose import ColumnTransformer
from sklearn.metrics import f1_score, make_scorer
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

MULTIMODAL_TEST_SIZE = 0.20
TUNING_CV_FOLDS = 3
TUNING_ITERATIONS = 8
CLIP_PCA_COMPONENTS = 64
MULTIMODAL_METRICS_CSV = Path('multimodal_xgboost_metrics.csv')
MULTIMODAL_TUNING_CSV = Path('multimodal_xgboost_tuning.csv')
MULTIMODAL_IMPORTANCE_CSV = Path('multimodal_xgboost_feature_importance.csv')
MULTIMODAL_IMPORTANCE_PNG = Path('multimodal_xgboost_feature_importance.png')

CLIP_FEATURES_CSV = Path('clip_visual_features.csv')
clip_features_df = pd.read_csv(CLIP_FEATURES_CSV)
cisf_features_df = pd.read_csv(Path('cisf_features.csv'))
CISF_COMPONENT_COLUMNS = [
    column for column in cisf_features_df.columns
    if column.startswith('cisf_fused_pca_')
]
assert len(CISF_COMPONENT_COLUMNS) == 32, (
    'Expected 32 CISF fused components. Re-run Step 8 first.'
)
CLIP_COLUMNS = [
    column for column in clip_features_df.columns
    if column.startswith('clip_') and column[5:].isdigit()
]
assert len(CLIP_COLUMNS) == 512, 'Expected 512 CLIP embedding features.'

multimodal_df = (
    text_df
    .merge(
        cisf_features_df[
            ['image_path', 'cisf_neighbour_mean_similarity', 'cisf_neighbour_max_similarity']
            + CISF_COMPONENT_COLUMNS
        ],
        on='image_path',
        how='inner',
    )
    .merge(
        clip_features_df[['image_path', 'clip_error'] + CLIP_COLUMNS],
        on='image_path',
        how='inner',
    )
)
multimodal_df['has_ocr_text'] = (
    multimodal_df['ocr_text'].fillna('').astype(str).str.strip().ne('').astype(float)
)

multimodal_df = multimodal_df.loc[
    multimodal_df['clip_error'].fillna('').eq('')
].copy().reset_index(drop=True)

assert multimodal_df['label'].nunique() == 2, 'Both classes are required.'
assert len(multimodal_df) == len(text_df), (
    'CLIP rows are missing. Re-run Step 12 until all feature rows are saved.'
)

for column in [
    'similarity_score', 'ocr_length', 'caption_length', 'has_ocr_text',
    'token_count', 'unique_token_ratio',
    'cisf_neighbour_mean_similarity', 'cisf_neighbour_max_similarity',
]:
    multimodal_df[column] = pd.to_numeric(
        multimodal_df[column], errors='coerce'
    ).fillna(0.0)

TABULAR_FEATURES = [
    'similarity_score',
    'ocr_length',
    'caption_length',
    'has_ocr_text',
    'token_count',
    'unique_token_ratio',
    'cisf_neighbour_mean_similarity',
    'cisf_neighbour_max_similarity',
]
TABULAR_FEATURES.extend(CISF_COMPONENT_COLUMNS)
for column in CISF_COMPONENT_COLUMNS:
    multimodal_df[column] = pd.to_numeric(
        multimodal_df[column], errors='coerce'
    ).fillna(0.0)

train_data, test_data = train_test_split(
    multimodal_df,
    test_size=MULTIMODAL_TEST_SIZE,
    stratify=multimodal_df['label'],
    random_state=SEED,
)

# A review-friendly copy of the exact images reserved for final testing.
HELDOUT_TEST_CSV = Path('multimodal_heldout_test_set.csv')
HELDOUT_EXPORT_COLUMNS = [
    'image_path', 'label', 'caption', 'ocr_text',
    'similarity_score', 'ocr_length', 'caption_length',
    'token_count', 'unique_token_ratio',
    'cisf_neighbour_mean_similarity', 'cisf_neighbour_max_similarity',
] + CISF_COMPONENT_COLUMNS
heldout_test_export = test_data[HELDOUT_EXPORT_COLUMNS].copy()
heldout_test_export.insert(0, 'dataset_split', 'held_out_test')
heldout_test_export.to_csv(HELDOUT_TEST_CSV, index=False)
y_train_multimodal = train_data['label'].eq('fake').astype(int)
y_test_multimodal = test_data['label'].eq('fake').astype(int)

multimodal_transformer = ColumnTransformer(
    transformers=[
        (
            'tfidf',
            TfidfVectorizer(
                tokenizer=str.split,
                token_pattern=None,
                lowercase=False,
                ngram_range=(1, 2),
                min_df=TEXT_MIN_DOCUMENT_FREQUENCY,
                max_features=TEXT_MAX_FEATURES,
                sublinear_tf=True,
                dtype=np.float32,
            ),
            'processed_text',
        ),
        ('tabular', 'passthrough', TABULAR_FEATURES),
        (
            'clip_pca',
            PCA(n_components=CLIP_PCA_COMPONENTS, random_state=SEED),
            CLIP_COLUMNS,
        ),
    ],
    sparse_threshold=0.30,
)

multimodal_xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=SEED,
    n_jobs=4,
    tree_method='hist',
    importance_type='gain',
)
multimodal_pipeline = Pipeline([
    ('features', multimodal_transformer),
    ('xgb', multimodal_xgb),
])

parameter_space = {
    'xgb__n_estimators': randint(150, 451),
    'xgb__max_depth': randint(3, 7),
    'xgb__learning_rate': loguniform(0.02, 0.15),
    'xgb__min_child_weight': randint(1, 7),
    'xgb__subsample': uniform(0.65, 0.35),
    'xgb__colsample_bytree': uniform(0.55, 0.40),
    'xgb__gamma': uniform(0.0, 1.0),
    'xgb__reg_alpha': loguniform(1e-5, 1e-1),
    'xgb__reg_lambda': loguniform(0.5, 5.0),
}

tuning_cv = StratifiedKFold(
    n_splits=TUNING_CV_FOLDS,
    shuffle=True,
    random_state=SEED,
)
fake_f1_scorer = make_scorer(f1_score, pos_label=1, zero_division=0)

tuned_multimodal_model = RandomizedSearchCV(
    estimator=multimodal_pipeline,
    param_distributions=parameter_space,
    n_iter=TUNING_ITERATIONS,
    scoring=fake_f1_scorer,
    cv=tuning_cv,
    random_state=SEED,
    n_jobs=1,
    refit=True,
    verbose=1,
)
tuned_multimodal_model.fit(train_data, y_train_multimodal)

tuning_results = pd.DataFrame(tuned_multimodal_model.cv_results_).sort_values(
    'rank_test_score'
)
tuning_results[
    ['rank_test_score', 'mean_test_score', 'std_test_score', 'params']
].to_csv(MULTIMODAL_TUNING_CSV, index=False)

multimodal_predictions_binary = tuned_multimodal_model.predict(test_data)
multimodal_predictions = np.where(
    multimodal_predictions_binary == 1, 'fake', 'real'
)
multimodal_actual = np.where(y_test_multimodal == 1, 'fake', 'real')

# Add per-image predictions and the overall score to the same 400-row CSV.
heldout_test_export['predicted_label'] = multimodal_predictions
heldout_test_export['correct_prediction'] = (
    heldout_test_export['label'].eq(heldout_test_export['predicted_label'])
)
heldout_test_export['overall_test_accuracy_percent'] = (
    accuracy_score(multimodal_actual, multimodal_predictions) * 100
)
heldout_test_export.to_csv(HELDOUT_TEST_CSV, index=False)

multimodal_metrics = pd.DataFrame([
    calculate_metrics(
        'Tuned multimodal XGBoost (TF-IDF + numeric + CISF + CLIP)',
        multimodal_actual,
        multimodal_predictions,
    )
])
multimodal_metrics['best_cv_f1_fake'] = tuned_multimodal_model.best_score_
multimodal_metrics['training_rows'] = len(train_data)
multimodal_metrics['test_rows'] = len(test_data)
multimodal_metrics.to_csv(MULTIMODAL_METRICS_CSV, index=False)

best_pipeline = tuned_multimodal_model.best_estimator_
best_feature_names = best_pipeline.named_steps['features'].get_feature_names_out()
best_importance = best_pipeline.named_steps['xgb'].feature_importances_
multimodal_importance = pd.DataFrame({
    'feature': best_feature_names,
    'gain_importance': best_importance,
}).sort_values('gain_importance', ascending=False).reset_index(drop=True)
multimodal_importance.to_csv(MULTIMODAL_IMPORTANCE_CSV, index=False)

top_multimodal_importance = multimodal_importance.head(25).sort_values('gain_importance')
fig, axis = plt.subplots(figsize=(10, 8))
axis.barh(
    top_multimodal_importance['feature'],
    top_multimodal_importance['gain_importance'],
    color='#0F766E',
)
axis.set_title('Tuned multimodal XGBoost: top 25 feature importances')
axis.set_xlabel('Relative gain importance')
fig.tight_layout()
fig.savefig(MULTIMODAL_IMPORTANCE_PNG, dpi=160, bbox_inches='tight')
plt.show()

print(f'Best 3-fold CV F1 (fake): {tuned_multimodal_model.best_score_:.4f}')
print(f'Best parameters: {tuned_multimodal_model.best_params_}')
print(multimodal_metrics.to_string(index=False, float_format=lambda value: f'{value:.4f}'))
print(f'\nMetrics saved to: {MULTIMODAL_METRICS_CSV.resolve()}')
print(f'Tuning results saved to: {MULTIMODAL_TUNING_CSV.resolve()}')
print(f'Feature importance saved to: {MULTIMODAL_IMPORTANCE_CSV.resolve()}')
print(f'Held-out test data saved to: {HELDOUT_TEST_CSV.resolve()}')


## Step 14 — Run the verification application and future modules

The production application is in `app.py`. It adds a local image-upload interface, the trained multimodal classifier, ChromaDB RAG, optional Serper live retrieval, optional Gemini + Groq evidence review, transparent social/source signals, and a human-approved dynamic update queue. API keys are never written in this notebook; put them in a private `.env` file using `.env.example` as the template.

In PowerShell, run `python scripts\train_production_model.py` once, then `streamlit run app.py`. See `README.md` for RAG indexing, dynamic updates, and independent-test steps.


In [ ]:
# Optional notebook check: confirms which live services are configured.
from news_guard.service import NewsVerificationService

verification_service = NewsVerificationService()
verification_service.service_status()


## Interpret the result

This is a baseline, not a fact-checking system. A strong score can reflect differences in data sources, templates, watermarking, or the image collection rather than genuine truthfulness. Before relying on it, inspect failed rows and OCR/captions, try a source-separated validation split, and evaluate on truly unseen sources.

**Optional exercise:** change the test split seed, rerun Step 7, and compare the four metrics. Large swings suggest the dataset or features are not yet stable enough.